<a href="https://colab.research.google.com/github/sinemdurmaz/Agent/blob/main/Normal_mi%3F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [101]:
#1#
!pip -q install -U \
    transformers \
    accelerate \
    "bitsandbytes>=0.46.1" \
    sentencepiece \
    gliner \
    langchain \
    langchain-community \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers

In [102]:
#2#
# ============================================================
# IMPORTS AND GOOGLE DRIVE
# ============================================================

import json
import re
from pathlib import Path

import torch

from google.colab import drive
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from gliner import GLiNER

from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader
)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

drive.mount("/content/drive")

print("Google Drive bağlandı.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive bağlandı.


In [103]:
#3#

PROJECT_DIR = Path("/content/drive/MyDrive/NORMALMI-AI")
RAG_DIR = PROJECT_DIR / "rag" / "pdfs"
PROMPT_PATH = PROJECT_DIR / "assistant" / "Llama_System_Instructions.md"

print("Proje klasörü var mı?:", PROJECT_DIR.exists())
print("RAG klasörü var mı?:", RAG_DIR.exists())
print("Prompt dosyası var mı?:", PROMPT_PATH.exists())

Proje klasörü var mı?: True
RAG klasörü var mı?: True
Prompt dosyası var mı?: True


In [104]:
#4#

rag_files = sorted(RAG_DIR.glob("*.md"))

print("Bulunan belge sayısı:", len(rag_files))

for file_path in rag_files:
    print("-", file_path.name)

Bulunan belge sayısı: 8
- 01_Preeclampsia.md
- 02_Warning_Signs.md
- 03_Bleeding.md
- 04_Fetal_Movement.md
- 05_Labor_Signs.md
- 06_Common_Symptoms.md
- 07_Glossary.md
- 08_System_Policies.md


In [105]:
#5#

PROJECT_DIR = Path("/content/drive/MyDrive/NORMALMI-AI")
ASSISTANT_DIR = PROJECT_DIR / "assistant"
PROMPT_PATH = ASSISTANT_DIR / "Llama_System_Instructions.md"

ASSISTANT_DIR.mkdir(parents=True, exist_ok=True)

if not PROMPT_PATH.exists():
    raise FileNotFoundError(
        f"System prompt dosyası bulunamadı: {PROMPT_PATH}"
    )

SYSTEM_PROMPT = PROMPT_PATH.read_text(
    encoding="utf-8"
)

print("System prompt yüklendi.")
print("Prompt yolu:", PROMPT_PATH)
print(SYSTEM_PROMPT[:500])

System prompt yüklendi.
Prompt yolu: /content/drive/MyDrive/NORMALMI-AI/assistant/Llama_System_Instructions.md
# ROL

Sen, KaraLabs Mother & Child Platformu için çalışan yerel bir gebelik semptom çıkarımı ve triyaj asistanısın.

Teşhis koyma.
İlaç önerme.
Tedavi önerme.
Kullanıcının söylemediği hiçbir belirtiyi ekleme.

# KRİTİK ÇALIŞMA KURALI

Risk kararını yalnızca sana verilen RAG referans belgelerindeki "Triyaj Kuralları" bölümlerine göre ver.

Kendi tıbbi bilgini kullanma.
Belgede bulunmayan bir risk kuralı üretme.
Birden fazla kural tetiklenirse en yüksek risk seviyesini seç.

Risk önceliği:

KIRMI


In [106]:
#6#

# ============================================================
# HUGGING FACE LOGIN
# ============================================================

login()

In [107]:
#8#

# ============================================================
# LOAD LLAMA 3.1 8B IN 4-BIT
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"
model.eval()

print("4-bit yüklendi:", getattr(model, "is_loaded_in_4bit", False))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


4-bit yüklendi: True


In [108]:
#10#

# ============================================================
# GLiNER MODEL
# ============================================================

model_ner = GLiNER.from_pretrained(
    "urchade/gliner_multi-v2.1"
)

print("GLiNER yüklendi.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

GLiNER yüklendi.


In [109]:
#11#

# ============================================================
# REGEX PATTERNS
# ============================================================

PHONE_PATTERN = r"\b(?:0?5\d{9})\b"

EMAIL_PATTERN = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

TC_PATTERN = r"\b\d{11}\b"

AGE_PATTERN = (
    r"\b(?:"
    r"\d{1,3}\s*(?:yaşında(?:yım|yim|sın|dır|dir)?|yaş)"
    r"|yaşım\s*\d{1,3}"
    r")\b"
)

In [110]:
#12#
# ============================================================
# MASKING FUNCTION — SAFE VERSION
# ============================================================

def mask_text(text: str) -> str:

    # GLiNER'ın değiştiremeyeceği geçici işaretler
    placeholders = {
        "__PII_PHONE__": "[PHONE]",
        "__PII_EMAIL__": "[EMAIL]",
        "__PII_TC_ID__": "[TC_ID]",
        "__PII_AGE__": "[AGE]"
    }

    # 1. Yapılandırılmış PII alanlarını regex ile koru
    text = re.sub(
        PHONE_PATTERN,
        "__PII_PHONE__",
        text
    )

    text = re.sub(
        EMAIL_PATTERN,
        "__PII_EMAIL__",
        text
    )

    text = re.sub(
        TC_PATTERN,
        "__PII_TC_ID__",
        text
    )

    text = re.sub(
        AGE_PATTERN,
        "__PII_AGE__",
        text
    )

    # 2. İsim ve konumları GLiNER ile bul
    labels = [
        "person",
        "location"
    ]

    entities = model_ner.predict_entities(
        text,
        labels,
        threshold=0.5
    )

    entities = sorted(
        entities,
        key=lambda x: x["start"],
        reverse=True
    )

    for ent in entities:

        entity_text = text[
            ent["start"]:ent["end"]
        ]

        # Geçici PII kodlarını değiştirme
        if "__PII_" in entity_text:
            continue

        if ent["label"] == "person":
            token = "[NAME]"

        elif ent["label"] == "location":
            token = "[LOCATION]"

        else:
            continue

        text = (
            text[:ent["start"]]
            + token
            + text[ent["end"]:]
        )

    # 3. Geçici kodları gerçek maskelere geri çevir
    for placeholder, token in placeholders.items():
        text = text.replace(
            placeholder,
            token
        )

    return text

In [111]:
#13#

# ============================================================
# LOAD RAG DOCUMENTS
# ============================================================

loader = DirectoryLoader(
    str(RAG_DIR),
    glob="*.md",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Yüklenen belge sayısı: {len(documents)}")

Yüklenen belge sayısı: 8


In [112]:
#14#
# ============================================================
# USE WHOLE DOCUMENTS
# ============================================================

docs = documents

print(f"Kullanılan belge sayısı: {len(docs)}")

Kullanılan belge sayısı: 8


In [113]:
#15#
# ============================================================
# EMBEDDING MODEL
# ============================================================

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Embedding modeli yüklendi.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding modeli yüklendi.


In [114]:
#16#
# ============================================================
# CREATE VECTOR DATABASE
# ============================================================

vectorstore = FAISS.from_documents(
    docs,
    embedding
)

print("FAISS oluşturuldu.")

FAISS oluşturuldu.


In [115]:
#17#
# ============================================================
# CREATE RETRIEVER
# ============================================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Retriever hazır.")

Retriever hazır.


In [116]:
#18#
# ============================================================
# USER INPUT
# ============================================================

user_text = input("Şikayetinizi yazınız:\n")

print("\nKULLANICI MESAJI")
print(user_text)

Şikayetinizi yazınız:
Ayşe Yılmaz yaşım 24.İzmirde yaşıyorum. 23 haftalık hamileyim suyum geldi tc no 12312312312, 05301231212. aaaaaaaaaa@gmail.com

KULLANICI MESAJI
Ayşe Yılmaz yaşım 24.İzmirde yaşıyorum. 23 haftalık hamileyim suyum geldi tc no 12312312312, 05301231212. aaaaaaaaaa@gmail.com


In [117]:
#19#
# ============================================================
# PII MASKING
# ============================================================

masked_text = mask_text(user_text)

print("\nMASKELENMİŞ METİN")
print(masked_text)


MASKELENMİŞ METİN
[NAME] [AGE].[LOCATION] yaşıyorum. 23 haftalık hamileyim suyum geldi tc no [TC_ID], [PHONE]. [EMAIL]


In [118]:
#26#
# ============================================================
# STEP 1 — SYMPTOM EXTRACTION
# ============================================================

EXTRACTION_PROMPT = """
Görevin yalnızca USER_MESSAGE içindeki açık bilgileri JSON alanlarına çıkarmaktır.

Riski belirleme.
Tıbbi yorum yapma.
Teşhis koyma.
Kullanıcının söylemediği hiçbir bilgiyi ekleme.

Kurallar:

- "23 haftalık hamileyim" -> gebelik_haftasi = 23
- "kanamam var" -> kanama_var_mi = true
- Kanama miktarı söylenmediyse kanama_miktari = null
- Kanama rengi söylenmediyse kanama_rengi = null
- "sancım var" tek başına düzenli kasılma değildir
- Düzenlilik veya dakika bilgisi yoksa duzenli_kasilma_var_mi = false
- Karın ağrısı açıkça söylenmediyse karin_agrisi_var_mi = false
- Kullanıcının söylediği en az bir klinik bilgi varsa anlasildi_mi = true
- Başarılı senaryoda kullanici_mesaji = null

Yalnızca tek bir geçerli JSON üret.
Markdown ve açıklama kullanma.

Şema:

{
  "semptom_ozeti": null,
  "sure": null,
  "gebelik_haftasi": null,
  "kanama_var_mi": false,
  "kanama_miktari": null,
  "kanama_rengi": null,
  "siddetli_bas_agrisi": false,
  "gorme_bozuklugu": false,
  "el_yuz_sisligi_ani": false,
  "duzenli_kasilma_var_mi": false,
  "kasilma_sikligi_dakika": null,
  "su_gelmesi_var_mi": false,
  "bebek_hareketi_azaldi_mi": false,
  "ates_var_mi": false,
  "ates_derece": null,
  "karin_agrisi_var_mi": false,
  "karin_agrisi_siddeti": 0,
  "karin_agrisi_sure_saat": 0,
  "bulanti": false,
  "kusma": false,
  "anlasildi_mi": true,
  "kullanici_mesaji": null
}
"""

extraction_messages = [
    {
        "role": "system",
        "content": EXTRACTION_PROMPT
    },
    {
        "role": "user",
        "content": f"""
<USER_MESSAGE>
{masked_text}
</USER_MESSAGE>
"""
    }
]

In [119]:
#27#
EXTRACTION_FALLBACK = {
    "semptom_ozeti": None,
    "sure": None,
    "gebelik_haftasi": None,
    "kanama_var_mi": False,
    "kanama_miktari": None,
    "kanama_rengi": None,
    "siddetli_bas_agrisi": False,
    "gorme_bozuklugu": False,
    "el_yuz_sisligi_ani": False,
    "duzenli_kasilma_var_mi": False,
    "kasilma_sikligi_dakika": None,
    "su_gelmesi_var_mi": False,
    "bebek_hareketi_azaldi_mi": False,
    "ates_var_mi": False,
    "ates_derece": None,
    "karin_agrisi_var_mi": False,
    "karin_agrisi_siddeti": 0,
    "karin_agrisi_sure_saat": 0,
    "bulanti": False,
    "kusma": False,
    "anlasildi_mi": False,
    "kullanici_mesaji": (
        "Gebelikle ilgili şikayetinizi biraz daha açık ifade eder misiniz?"
    )
}

In [120]:
#28#
def parse_extraction_json(response_text: str) -> dict:
    try:
        match = re.search(r"\{[\s\S]*?\}", response_text)

        if not match:
            raise ValueError("Semptom JSON'u bulunamadı.")

        parsed = json.loads(match.group())

        required_keys = set(EXTRACTION_FALLBACK.keys())
        parsed_keys = set(parsed.keys())

        missing_keys = required_keys - parsed_keys
        unexpected_keys = parsed_keys - required_keys

        if missing_keys:
            raise ValueError(
                f"Eksik alanlar: {sorted(missing_keys)}"
            )

        if unexpected_keys:
            raise ValueError(
                f"Beklenmeyen alanlar: {sorted(unexpected_keys)}"
            )

        return parsed

    except (
        json.JSONDecodeError,
        TypeError,
        ValueError,
        KeyError
    ) as error:
        print("Semptom JSON doğrulama hatası:", error)
        return EXTRACTION_FALLBACK.copy()

In [121]:
#29#
# ============================================================
# LOCAL EXTRACTION GUARD
# ============================================================

def guard_extracted_symptoms(
    veri: dict,
    user_message: str
) -> dict:

    text = user_message.casefold()

    # --------------------------------------------------------
    # GEBELİK HAFTASI
    # --------------------------------------------------------

    week_match = re.search(
        r"\b(\d{1,2})\s*haftalık\b",
        text
    )

    if week_match:
        veri["gebelik_haftasi"] = int(
            week_match.group(1)
        )
    else:
        veri["gebelik_haftasi"] = None

    # --------------------------------------------------------
    # KANAMA
    # --------------------------------------------------------

    bleeding_words = [
        "kanama",
        "kanamam",
        "kan geliyor",
        "kan geldi",
        "kanıyorum",
        "lekelenme",
        "lekelenmem"
    ]

    has_bleeding = any(
        word in text
        for word in bleeding_words
    )

    veri["kanama_var_mi"] = has_bleeding

    if not has_bleeding:
        veri["kanama_miktari"] = None
        veri["kanama_rengi"] = None

    # Kanama miktarı sadece açıkça belirtilmişse korunur.
    amount_words = [
        "az kanama",
        "hafif kanama",
        "yoğun kanama",
        "çok kanama",
        "fazla kanama",
        "birkaç damla"
    ]

    if not any(word in text for word in amount_words):
        veri["kanama_miktari"] = None

    # Kanama rengi sadece açıkça belirtilmişse korunur.
    color_words = [
        "kırmızı kan",
        "kahverengi kan",
        "kahverengi leke",
        "pembe kan",
        "pembe leke",
        "koyu renkli kan"
    ]

    if not any(word in text for word in color_words):
        veri["kanama_rengi"] = None

    # --------------------------------------------------------
    # SU GELMESİ
    # --------------------------------------------------------

    water_words = [
        "suyum geldi",
        "suyum boşaldı",
        "su geliyor",
        "vajinamdan su geliyor",
        "berrak sıvı geliyor",
        "sürekli ıslanıyorum"
    ]

    veri["su_gelmesi_var_mi"] = any(
        word in text
        for word in water_words
    )

    # --------------------------------------------------------
    # DÜZENLİ KASILMA
    # --------------------------------------------------------

    contraction_match = re.search(
        r"\b(\d+)\s*dakikada bir\s+"
        r"(?:sancı|kasılma)",
        text
    )

    regular_contraction = (
        "düzenli kasıl" in text
        or "düzenli sancı" in text
        or "sancılarım giderek sıklaşıyor" in text
        or "kasılmalarım giderek sıklaşıyor" in text
        or contraction_match is not None
    )

    veri["duzenli_kasilma_var_mi"] = bool(
        regular_contraction
    )

    if contraction_match:
        veri["kasilma_sikligi_dakika"] = int(
            contraction_match.group(1)
        )
    else:
        veri["kasilma_sikligi_dakika"] = None

    # --------------------------------------------------------
    # KARIN AĞRISI
    # --------------------------------------------------------

    abdominal_pain_words = [
        "karın ağrısı",
        "karın ağrım",
        "karnım ağrıyor",
        "karın sancısı",
        "kramp",
        "kasık ağrısı"
    ]

    has_abdominal_pain = any(
        word in text
        for word in abdominal_pain_words
    )

    veri["karin_agrisi_var_mi"] = has_abdominal_pain

    # Kullanıcı sayısal şiddet vermediyse 0.
    pain_score_match = re.search(
        r"(?:ağrı|sancı).*?"
        r"\b([0-9]|10)\s*(?:/|üzerinden)\s*10\b",
        text
    )

    if has_abdominal_pain and pain_score_match:
        veri["karin_agrisi_siddeti"] = int(
            pain_score_match.group(1)
        )
    else:
        veri["karin_agrisi_siddeti"] = 0

    # Kullanıcı saat olarak süre vermediyse 0.
    pain_duration_match = re.search(
        r"\b(\d+(?:[.,]\d+)?)\s*saattir\b",
        text
    )

    if has_abdominal_pain and pain_duration_match:
        veri["karin_agrisi_sure_saat"] = float(
            pain_duration_match.group(1).replace(",", ".")
        )
    else:
        veri["karin_agrisi_sure_saat"] = 0

    # --------------------------------------------------------
    # ŞİDDETLİ BAŞ AĞRISI
    # --------------------------------------------------------

    headache_words = [
        "şiddetli baş ağrısı",
        "başım çok ağrıyor",
        "başım çatlıyor",
        "dayanılmaz baş ağrısı",
        "başım şiddetli ağrıyor"
    ]

    veri["siddetli_bas_agrisi"] = any(
        word in text
        for word in headache_words
    )

    # --------------------------------------------------------
    # GÖRME BOZUKLUĞU
    # --------------------------------------------------------

    vision_words = [
        "bulanık görüyorum",
        "bulanık görme",
        "ışık çakması",
        "ışık çakıyor",
        "ışıklar görüyorum",
        "gözümün önü kararıyor",
        "gözüm kararıyor",
        "net göremiyorum",
        "siyah noktalar görüyorum"
    ]

    veri["gorme_bozuklugu"] = any(
        word in text
        for word in vision_words
    )

    # --------------------------------------------------------
    # ANİ EL VEYA YÜZ ŞİŞLİĞİ
    # --------------------------------------------------------

    swelling_words = [
        "ellerim aniden şişti",
        "yüzüm aniden şişti",
        "elim yüzüm şişti",
        "el ve yüzüm şişti",
        "ani şişlik",
        "göz kapaklarım şişti"
    ]

    veri["el_yuz_sisligi_ani"] = any(
        word in text
        for word in swelling_words
    )

    # --------------------------------------------------------
    # BEBEK HAREKETLERİNDE AZALMA
    # --------------------------------------------------------

    fetal_movement_words = [
        "bebek hareketleri azaldı",
        "bebeğimin hareketleri azaldı",
        "bebeğim hareket etmiyor",
        "eskisi kadar hareket etmiyor",
        "tekme atmıyor",
        "hiç kıpırdamıyor"
    ]

    veri["bebek_hareketi_azaldi_mi"] = any(
        word in text
        for word in fetal_movement_words
    )

    # --------------------------------------------------------
    # ATEŞ
    # --------------------------------------------------------

    fever_words = [
        "ateşim var",
        "yüksek ateşim var",
        "ateşim çıktı"
    ]

    veri["ates_var_mi"] = any(
        word in text
        for word in fever_words
    )

    fever_match = re.search(
        r"\b(\d{2}(?:[.,]\d)?)\s*derece\b",
        text
    )

    if fever_match:
        veri["ates_var_mi"] = True
        veri["ates_derece"] = float(
            fever_match.group(1).replace(",", ".")
        )
    else:
        veri["ates_derece"] = None

    # --------------------------------------------------------
    # BULANTI
    # --------------------------------------------------------

    nausea_words = [
        "bulantım var",
        "midem bulanıyor",
        "mide bulantısı"
    ]

    veri["bulanti"] = any(
        word in text
        for word in nausea_words
    )

    # --------------------------------------------------------
    # KUSMA
    # --------------------------------------------------------

    vomiting_words = [
        "kusuyorum",
        "kustum",
        "sürekli kusuyorum",
        "kusmam var"
    ]

    veri["kusma"] = any(
        word in text
        for word in vomiting_words
    )

    # --------------------------------------------------------
    # ANLAŞILDI MI?
    # --------------------------------------------------------

    symptom_flags = [
        veri["kanama_var_mi"],
        veri["su_gelmesi_var_mi"],
        veri["duzenli_kasilma_var_mi"],
        veri["karin_agrisi_var_mi"],
        veri["siddetli_bas_agrisi"],
        veri["gorme_bozuklugu"],
        veri["el_yuz_sisligi_ani"],
        veri["bebek_hareketi_azaldi_mi"],
        veri["ates_var_mi"],
        veri["bulanti"],
        veri["kusma"]
    ]

    clinical_info = (
        veri["gebelik_haftasi"] is not None
        or any(symptom_flags)
    )

    veri["anlasildi_mi"] = bool(clinical_info)

    if clinical_info:
        veri["kullanici_mesaji"] = None
    else:
        veri["kullanici_mesaji"] = (
            "Gebelikle ilgili şikayetinizi "
            "biraz daha açık ifade eder misiniz?"
        )

    return veri


In [122]:
#30#
# ============================================================
# GENERATE EXTRACTION RESPONSE
# ============================================================

extraction_inputs = tokenizer.apply_chat_template(
    extraction_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    extraction_outputs = model.generate(
        **extraction_inputs,
        max_new_tokens=500,
        do_sample=False,
        use_cache=True,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

extraction_generated_tokens = extraction_outputs[0][
    extraction_inputs["input_ids"].shape[-1]:
]

extraction_response = tokenizer.decode(
    extraction_generated_tokens,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
).strip()

print("\nHAM SEMPTOM ÇIKARIMI")
print(extraction_response)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



HAM SEMPTOM ÇIKARIMI
### Kullanıcı Mesajı Analizi

#### Giriş

Kullanıcı mesajı analiz edilecek ve ilgili bilgiler JSON şemasına göre doldurulacaktır.

#### Kullanıcı Mesajı

```
[NAME] [AGE].[LOCATION] yaşıyorum. 23 haftalık hamileyim suyum geldi tc no [TC_ID], [PHONE]. [EMAIL]
```

#### Analiz

*   Kullanıcı adı, yaş, lokasyon, gebelik haftası, suyun gelmesi ve TC numarası ile telefon numarası ve email adresi gibi bilgiler mevcuttur.
*   Gebelik haftası 23 olarak belirtilmiştir.
*   Suyun gelmesi belirtildiği için su_gelmesi_var_mi = true olmalıdır.
*   Diğer symptomlar veya şikayetler belirtilmemiştir.

#### JSON Şeması

```json
{
  "semptom_ozeti": null,
  "sure": null,
  "gebelik_haftasi": 23,
  "kanama_var_mi": false,
  "kanama_miktari": null,
  "kanama_rengi": null,
  "siddetli_bas_agrisi": false,
  "gorme_bozuklugu": false,
  "el_yuz_sisligi_ani": false,
  "duzenli_kasilma_var_mi": false,
  "kasilma_sikligi_dakika": null,
  "su_gelmesi_var_mi": true,
  "bebek_hareketi_azaldi_m

In [123]:
#31#
extracted = parse_extraction_json(
    extraction_response
)

extracted = guard_extracted_symptoms(
    extracted,
    masked_text
)

print("\nDOĞRULANMIŞ SEMPTOM JSON")
print(
    json.dumps(
        extracted,
        ensure_ascii=False,
        indent=2
    )
)


DOĞRULANMIŞ SEMPTOM JSON
{
  "semptom_ozeti": null,
  "sure": null,
  "gebelik_haftasi": 23,
  "kanama_var_mi": false,
  "kanama_miktari": null,
  "kanama_rengi": null,
  "siddetli_bas_agrisi": false,
  "gorme_bozuklugu": false,
  "el_yuz_sisligi_ani": false,
  "duzenli_kasilma_var_mi": false,
  "kasilma_sikligi_dakika": null,
  "su_gelmesi_var_mi": true,
  "bebek_hareketi_azaldi_mi": false,
  "ates_var_mi": false,
  "ates_derece": null,
  "karin_agrisi_var_mi": false,
  "karin_agrisi_siddeti": 0,
  "karin_agrisi_sure_saat": 0,
  "bulanti": false,
  "kusma": false,
  "anlasildi_mi": true,
  "kullanici_mesaji": null
}


In [124]:
#37#
def build_retrieval_query(extracted: dict) -> str:
    query_parts = []

    if extracted.get("kanama_var_mi"):
        query_parts.append("gebelikte kanama")

    if extracted.get("su_gelmesi_var_mi"):
        query_parts.append("gebelikte su gelmesi")

    if extracted.get("duzenli_kasilma_var_mi"):
        query_parts.append("düzenli kasılma erken doğum")

    if extracted.get("bebek_hareketi_azaldi_mi"):
        query_parts.append("bebek hareketlerinde azalma")

    if (
        extracted.get("siddetli_bas_agrisi")
        or extracted.get("gorme_bozuklugu")
        or extracted.get("el_yuz_sisligi_ani")
    ):
        query_parts.append(
            "şiddetli baş ağrısı görme bozukluğu ani şişlik"
        )

    if extracted.get("ates_var_mi"):
        query_parts.append("gebelikte ateş")

    if extracted.get("bulanti") or extracted.get("kusma"):
        query_parts.append("gebelikte bulantı kusma")

    if extracted.get("karin_agrisi_var_mi"):
        query_parts.append("gebelikte karın ağrısı")

    return " ".join(query_parts).strip()

In [125]:
#38#
# ============================================================
# RAG RETRIEVAL FROM VALIDATED SYMPTOMS
# ============================================================

retrieval_query = build_retrieval_query(extracted)

print("\nRETRIEVAL SORGUSU")
print(
    retrieval_query
    if retrieval_query
    else "Triyaj için semptom bulunamadı."
)

if retrieval_query:
    retrieved_docs = retriever.invoke(retrieval_query)

    rag_context = "\n\n--------------------\n\n".join(
        f"""
BELGE: {Path(doc.metadata["source"]).name}

{doc.page_content}
""".strip()
        for doc in retrieved_docs
    )
else:
    retrieved_docs = []
    rag_context = ""

print("\nRAG REFERANSLARI")

if retrieved_docs:
    for i, doc in enumerate(retrieved_docs, 1):
        print(
            f"{i}. {Path(doc.metadata['source']).name}"
        )
else:
    print("İlgili RAG belgesi getirilmedi.")

print("\nRETRIEVAL SORGUSU:")
print(retrieval_query)

print("\nGETİRİLEN RAG İÇERİĞİ:")
print(rag_context)


RETRIEVAL SORGUSU
gebelikte su gelmesi

RAG REFERANSLARI
1. 05_Labor_Signs.md
2. 03_Bleeding.md

RETRIEVAL SORGUSU:
gebelikte su gelmesi

GETİRİLEN RAG İÇERİĞİ:
BELGE: 05_Labor_Signs.md

---
title: Labor Signs and Preterm Labor
category: Pregnancy Emergency
version: 4.0
language: tr
---

# Amaç

Bu belge, su gelmesi ve düzenli kasılma bildirimlerini anlamak ve triyaj kodunu belirlemek için kullanılır.

Bu belge teşhis koymaz.

# Ana Kullanıcı İfadeleri

## Su Gelmesi

Aşağıdaki ifadeler `su_gelmesi_var_mi = true` anlamına gelir:

- suyum geldi
- suyum boşaldı
- vajinamdan su geliyor
- berrak sıvı geliyor
- iç çamaşırım sürekli ıslanıyor
- amniyon sıvısı geliyor

## Düzenli Kasılma

Aşağıdaki ifadeler `duzenli_kasilma_var_mi = true` anlamına gelir:

- düzenli kasılmalarım var
- 10 dakikada bir sancım geliyor
- 5 dakikada bir sancım geliyor
- sancılar giderek sıklaşıyor
- belirli aralıklarla kasılıyorum
- doğum sancılarım düzenli

Aşağıdaki ifadeler tek başına düzenli kasılma değildir:


In [126]:
#39#
# ============================================================
# TRIAGE AVAILABILITY CONTROL
# ============================================================

triage_can_run = bool(
    retrieval_query
    and rag_context.strip()
)

In [127]:
#32#
# ============================================================
# STEP 2 — TRIAGE DECISION
# ============================================================

TRIAGE_PROMPT = """
Sen yalnızca triyaj kararı veren bir JSON motorusun.

VALIDATED_SYMPTOMS içindeki alanlar lokal Python koduyla doğrulanmıştır.
Bu alanları değiştirme ve yeni semptom ekleme.

Risk kararını yalnızca RETRIEVED_RULES içindeki Triyaj Kurallarına göre ver.

Uygulama sırası:

1. true olan semptom alanlarını belirle.
2. Bu semptomlarla eşleşen RAG triyaj kurallarını bul.
3. Birden fazla kural varsa en yüksek risk kodunu seç.

Risk önceliği:

KIRMIZI > SARI > YESIL

risk_kodu yalnızca şunlardan biri olabilir:

"KIRMIZI"
"SARI"
"YESIL"
"BELIRSIZ"

Önemli:

- su_gelmesi_var_mi=true ise RETRIEVED_RULES içindeki su gelmesi kuralını uygula.
- İlgili kural KIRMIZI diyorsa risk_kodu="KIRMIZI" üret.
- İlgili RAG kuralı yoksa risk_kodu="BELIRSIZ" üret.
- Teşhis koyma.
- Kendi tıbbi bilgini kullanma.
- Markdown kullanma.
- Kod bloğu oluşturma.
- JSON dışında açıklama yazma.

Yalnızca şu iki alanı içeren tek JSON nesnesi üret:

{
  "risk_kodu": "BELIRSIZ",
  "risk_gerekcesi": null
}
"""

In [128]:
#33#
triage_messages = [
    {
        "role": "system",
        "content": TRIAGE_PROMPT
    },
    {
        "role": "user",
        "content": f"""
<VALIDATED_SYMPTOMS>
{json.dumps(extracted, ensure_ascii=False, indent=2)}
</VALIDATED_SYMPTOMS>

<RETRIEVED_RULES>
{rag_context}
</RETRIEVED_RULES>

Yalnızca JSON üret.
"""
    }
]

In [129]:
#34#
# ============================================================
# GENERATE TRIAGE RESPONSE
# ============================================================

if not triage_can_run:

    triage_response = json.dumps(
        {
            "risk_kodu": "BELIRSIZ",
            "risk_gerekcesi": None
        },
        ensure_ascii=False
    )

    print(
        "\nTriyaj için eşleşen semptom veya "
        "RAG kuralı bulunamadı."
    )

else:

    triage_inputs = tokenizer.apply_chat_template(
        triage_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    print(
        "Triyaj girdi token sayısı:",
        triage_inputs["input_ids"].shape[-1]
    )

    with torch.inference_mode():
        triage_outputs = model.generate(
            **triage_inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    prompt_length = triage_inputs["input_ids"].shape[-1]

    triage_generated_tokens = triage_outputs[0][
        prompt_length:
    ]

    triage_response = tokenizer.decode(
        triage_generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

print("\nHAM TRİYAJ ÇIKTISI")
print(repr(triage_response))

Triyaj girdi token sayısı: 2675

HAM TRİYAJ ÇIKTISI
'{\n  "risk_kodu": "KIRMIZI",\n  "risk_gerekcesi": "su_gelmesi_var_mi = true"\n}'


In [130]:
#35#
TRIAGE_FALLBACK = {
    "risk_kodu": "BELIRSIZ",
    "risk_gerekcesi": None
}


def parse_triage_json(response_text: str) -> dict:

    try:
        if not isinstance(response_text, str):
            raise TypeError(
                "Triyaj yanıtı metin olmalıdır."
            )

        cleaned_response = response_text.strip()

        if not cleaned_response:
            raise ValueError(
                "Triyaj modeli boş yanıt üretti."
            )

        match = re.search(
            r"\{[\s\S]*?\}",
            cleaned_response
        )

        if not match:
            raise ValueError(
                "Risk JSON'u bulunamadı."
            )

        parsed = json.loads(
            match.group()
        )

        if not isinstance(parsed, dict):
            raise TypeError(
                "Triyaj çıktısı JSON nesnesi değildir."
            )

        required_keys = {
            "risk_kodu",
            "risk_gerekcesi"
        }

        missing_keys = required_keys - set(
            parsed.keys()
        )

        if missing_keys:
            raise ValueError(
                f"Eksik triyaj alanları: "
                f"{sorted(missing_keys)}"
            )

        allowed = {
            "KIRMIZI",
            "SARI",
            "YESIL",
            "BELIRSIZ"
        }

        if parsed["risk_kodu"] not in allowed:
            raise ValueError(
                f"Geçersiz risk kodu: "
                f"{parsed['risk_kodu']}"
            )

        return {
            "risk_kodu": parsed["risk_kodu"],
            "risk_gerekcesi": parsed[
                "risk_gerekcesi"
            ]
        }

    except (
        json.JSONDecodeError,
        TypeError,
        ValueError,
        KeyError
    ) as error:

        print("\nRisk JSON doğrulama hatası:")
        print(error)
        print("Triyaj fallback JSON döndürülüyor.")

        return TRIAGE_FALLBACK.copy()

In [131]:
#36#
# ============================================================
# BUILD AND VALIDATE FINAL JSON
# ============================================================

FINAL_FALLBACK = {
    **EXTRACTION_FALLBACK,
    **TRIAGE_FALLBACK
}


def build_final_json(
    extracted_data: dict,
    triage_data: dict
) -> dict:

    try:
        final_data = {
            **extracted_data,
            **triage_data
        }

        required_keys = set(FINAL_FALLBACK.keys())
        final_keys = set(final_data.keys())

        missing_keys = required_keys - final_keys
        unexpected_keys = final_keys - required_keys

        if missing_keys:
            raise ValueError(
                f"Nihai JSON eksik alanları: {sorted(missing_keys)}"
            )

        if unexpected_keys:
            raise ValueError(
                f"Nihai JSON beklenmeyen alanları: "
                f"{sorted(unexpected_keys)}"
            )

        if not isinstance(
            final_data["anlasildi_mi"],
            bool
        ):
            raise TypeError(
                "anlasildi_mi boolean olmalıdır."
            )

        if final_data["risk_kodu"] not in {
            "KIRMIZI",
            "SARI",
            "YESIL",
            "BELIRSIZ"
        }:
            raise ValueError(
                "Nihai JSON risk kodu geçersiz."
            )

        return final_data

    except (
        TypeError,
        ValueError,
        KeyError
    ) as error:

        print("\nNihai JSON doğrulama hatası:")
        print(error)
        print("Nihai fallback JSON döndürülüyor.")

        return FINAL_FALLBACK.copy()


triage_result = parse_triage_json(
    triage_response
)

final_json = build_final_json(
    extracted,
    triage_result
)

print("\nNİHAİ JSON")

print(
    json.dumps(
        final_json,
        ensure_ascii=False,
        indent=2
    )
)


NİHAİ JSON
{
  "semptom_ozeti": null,
  "sure": null,
  "gebelik_haftasi": 23,
  "kanama_var_mi": false,
  "kanama_miktari": null,
  "kanama_rengi": null,
  "siddetli_bas_agrisi": false,
  "gorme_bozuklugu": false,
  "el_yuz_sisligi_ani": false,
  "duzenli_kasilma_var_mi": false,
  "kasilma_sikligi_dakika": null,
  "su_gelmesi_var_mi": true,
  "bebek_hareketi_azaldi_mi": false,
  "ates_var_mi": false,
  "ates_derece": null,
  "karin_agrisi_var_mi": false,
  "karin_agrisi_siddeti": 0,
  "karin_agrisi_sure_saat": 0,
  "bulanti": false,
  "kusma": false,
  "anlasildi_mi": true,
  "kullanici_mesaji": null,
  "risk_kodu": "KIRMIZI",
  "risk_gerekcesi": "su_gelmesi_var_mi = true"
}
